# Simple cue-following evaluation

Question: does the model solve the math problem, or does it copy the bad clue from earlier in the conversation?

## 1. Choose where the model runs

- `dryrun`: no model needed; useful for explaining the flow.
- `ollama`: local model on your computer.
- `azure-openai`: Azure OpenAI compatible deployment.
- `openrouter`: hosted model through OpenRouter.
- `aws`: Amazon Bedrock model through Mantle API.
- `azure-foundry`: Microsoft Foundry model deployment.
- `azure-ai`: older Azure AI Model Inference endpoint.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

PROVIDER = "dryrun"
MODEL = "qwen3-32b"
DATA_PATH = ROOT / "data" / "math500_prepared_50.jsonl"
OUTPUT_DIR = ROOT / "outputs" / "notebook_demo"

## 2. Run the experiment

The three groups are: no prior bad-clue story turns, 2 bad-clue story turns, and 10 bad-clue story turns.

In [ ]:
from cue_eval.experiment import run_experiment

rows, summary = run_experiment(
    data_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
    provider=PROVIDER,
    model=MODEL,
    cue_counts=[0, 2, 10],
    temperature=0.2,
)

summary

## 3. Inspect a few answers

Each answer is labeled as `correct`, `followed_bad_clue`, `other_wrong_answer`, or `parse_fail`.

In [ ]:
for row in rows[:6]:
    print(row["id"], "cue_count=", row["cue_count"], "answer=", row["parsed_answer"], "label=", row["label"])
    print(row["response"][:250])
    print("---")

## 4. Show the chart

This is the main coach-facing result.

In [ ]:
from IPython.display import Image, display

chart_path = OUTPUT_DIR / "shortcut_rate.png"
if chart_path.exists():
    display(Image(filename=str(chart_path)))
else:
    print("Install matplotlib to create the chart: pip install -r requirements.txt")

## Optional: run with Ollama

In a terminal:

```powershell
ollama pull qwen3:14b
python scripts/run_experiment1.py --provider ollama --model qwen3:14b --prepared-dir data
```

Then set `PROVIDER = "ollama"` above and rerun the notebook cells.

## Optional: run with OpenRouter

Set these environment variables in a notebook cell before running:

```python
import os, getpass
os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")
PROVIDER = "openrouter"
MODEL = "qwen/qwen3-32b"
```

Then set `PROVIDER = "openrouter"` above and rerun the notebook cells.

## Optional: run with AWS Bedrock

Set these environment variables in a notebook cell before running:

```python
import os
os.environ["AWS_BEDROCK_REGION"] = "us-east-1"
os.environ["BEDROCK_API_KEY"] = "YOUR-BEDROCK-API-KEY"
PROVIDER = "aws"
MODEL = "us.anthropic.claude-3-5-haiku-20241022-v1:0"
```

Then set `PROVIDER = "aws"` above and rerun the notebook cells.

## Optional: run with Azure

For Microsoft Foundry, set these environment variables in a notebook cell before running:

```python
import os, getpass
os.environ["AZURE_AI_ENDPOINT"] = "https://YOUR-RESOURCE.services.ai.azure.com"
os.environ["AZURE_AI_API_KEY"] = getpass.getpass("Foundry API key: ")
PROVIDER = "azure-foundry"
MODEL = "qwen3-32b"
```

Then set `PROVIDER = "azure-foundry"` above and rerun the notebook cells.